# Semantic Kernel 

या कोड नमुन्यात, आपण एक सोपा एजंट तयार करण्यासाठी [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI फ्रेमवर्क वापराल.

या नमुन्याचा उद्देश म्हणजे विविध एजंटिक पॅटर्न्स अंमलात आणताना पुढील कोड नमुन्यांमध्ये लागू केले जाणारे टप्पे दाखवणे.


## आवश्यक पायथन पॅकेजेस आयात करा


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## क्लाएंट तयार करणे

या उदाहरणात, आपण LLM ला ऍक्सेस करण्यासाठी [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) वापरणार आहोत.

`ai_model_id` `gpt-4o-mini` म्हणून सेट केलेले आहे. GitHub Models मार्केटप्लेसमधील दुसऱ्या उपलब्ध मॉडेलवर स्विच करून वेगवेगळे परिणाम पहाण्याचा प्रयत्न करा.

GitHub Models साठी आवश्यक असलेल्या `base_url` साठी `Azure Inference SDK` वापरण्यासाठी, आपण Semantic Kernel मधील `OpenAIChatCompletion` कनेक्टरचा उपयोग करू. तसेच, Semantic Kernel वापरून अन्य मॉडेल प्रदात्यांसोबत वापरण्यासाठी इतर [उपलब्ध कनेक्टर्स](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) देखील आहेत.


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## एजंट तयार करणे

येथे, आम्ही `TravelAgent` नावाचा एजंट तयार करतो.

या उदाहरणात, आम्ही खूप मूलभूत सूचना वापरतो. एजंटच्या वर्तनात कशी बदल होते हे पाहण्यासाठी या सूचना वांछित प्रमाणे सुधारू शकता.


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## एजंट चालविणे

आता आपण `ChatHistory` सेट करून आणि त्यात `system_message` समाविष्ट करून एजंट चालवू शकतो. आपण पूर्वी स्थापित केलेले `AGENT_INSTRUCTIONS` वापरू.

हे सर्व सेट केल्यावर, आपण `user_inputs` तयार करतो, जे वापरकर्ता एजंटला काय पाठवत आहे हे दर्शवते. या उदाहरणात, संदेश `Plan me a sunny vacation` असे सेट केले आहे.

आपण हा संदेश बदलून पाहू शकता की एजंट वेगवेगळ्या प्रकारे कसे प्रतिसाद देते.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**जाहिरात**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा असमर्थता असू शकते. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहिती साठी, व्यावसायिक मानवी भाषांतर सुचविले जाते. या भाषांतराच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुतीबद्दल किंवा चुकीच्या अर्थ लावण्याबद्दल आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
